# Writer / Editor Hidden-State Probe

**Three-arm comparison of agent communication via hidden states**

| Arm | Key | What it does |
|-----|-----|--------------|
| A | `editor_verdict` | Editor receives the **raw essay text** in its prompt |
| B | `editor_selfhs_verdict` | Editor's **own final-layer hidden states** over the draft span are re-injected at `inject_layer` |
| C | `editor_writerhs_verdict` | The **writer's final-layer hidden states** (over its answer tokens) are injected into the editor at `inject_layer` |

Comparing A, B, C isolates where communication loss occurs between agents:
- **A ≈ B** → the editor's internal representation carries the same info as the surface text
- **A ≈ C** → the writer's hidden states carry the same info as its decoded text
- **B ≈ C** → writer and editor form compatible internal representations

**Total cost per run: 4 `generate()` calls** — 1 writer + 3 editor (A, B, C).

### Qwen3 / Qwen3.5 — thinking mode
Reasoning stays **ON** by default.  `ModelBackend` automatically strips `<think>…</think>` blocks from
`output_text` and `gen_token_ids` so verdicts are always clean text. Raw thinking is still accessible via `result.thinking_text`.

### Supported models

| Family | Example IDs | Notes |
|--------|------------|-------|
| **Qwen 3.5** | `Qwen/Qwen3.5-0.8B` · `Qwen/Qwen3.5-4B` · `Qwen/Qwen3.5-9B` · `Qwen/Qwen3.5-27B` · `Qwen/Qwen3.5-35B-A3B` (MoE) · `Qwen/Qwen3.5-122B-A10B` (MoE) | Thinking stripped automatically |
| **Qwen 2.5** | `Qwen/Qwen2.5-{0.5,1.5,3,7,14,32,72}B-Instruct` | Standard path |
| **LLaMA 3.x** | `meta-llama/Llama-3.3-70B-Instruct` | Standard path |
| **Gemma 3/4** | `google/gemma-3-{4b,12b,27b}-it` | Standard path |
| **K2-V2** | `LLM360/K2-V2-Instruct` | Use `K2V2Backend` for `reasoning_effort` |

## 0 — Install

In [ ]:
# Install the package + all ML dependencies from GitHub:
!pip install -q git+https://github.com/YOUR_USERNAME/reagent.git

# Or, if you cloned the repo locally:
# !pip install -q -e ..

# Qwen3.5 is a new architecture — install the latest transformers from source:
# (skip this line for older model families like Qwen2.5, LLaMA, Gemma)
!pip install -q --upgrade "git+https://github.com/huggingface/transformers.git@main"

# ───────────────────────────────────────────────────────────────────────────
# CRITICAL for Qwen 3.5 performance: install its linear-attention CUDA kernels.
# Without these, Qwen 3.5 falls back to a PURE-PYTHON reference implementation
# and a single run takes 15-20 minutes instead of ~1-2 minutes on an A100.
#
# If you're running a NON-Qwen-3.5 model (LLaMA 3.3, Gemma 3/4, Qwen 2.5, K2-V2),
# you can skip this cell — those use standard softmax attention.
# ───────────────────────────────────────────────────────────────────────────
!pip install -q flash-linear-attention
!pip install -q causal-conv1d
# Optional: flash-attention speeds up the softmax-attention layers Qwen 3.5 also has.
!pip install -q flash-attn --no-build-isolation || echo "flash-attn install failed (optional — continuing)" 

In [ ]:
import time
import torch

from selfie_k2v2 import (
    ModelBackend,
    K2V2Backend,
    HiddenStateProjector,
    make_graph,
    add_probe_to_graph,
    ProbeMixin,
)

## 1 — Choose your model(s)

Writer and editor can be the **same** instance (shared weights) or
**different** `ModelBackend` instances for cross-model experiments.

### 1a — Qwen 3.5 (all sizes)

Reasoning is **ON**. `<think>…</think>` blocks are stripped automatically from verdicts.
Access raw thinking via `final["writer_result"].thinking_text`.

| Model ID | Params | VRAM (bf16) | Notes |
|----------|--------|-------------|-------|
| `Qwen/Qwen3.5-0.8B` | 0.9 B | ~2 GB | dev / smoke-test |
| `Qwen/Qwen3.5-4B` | ~4 B | ~8 GB | single consumer GPU |
| `Qwen/Qwen3.5-9B` | ~9 B | ~18 GB | single A100-40 GB |
| `Qwen/Qwen3.5-27B` | 28 B | ~55 GB | 2× A100-40 GB or 1× H100 |
| `Qwen/Qwen3.5-35B-A3B` | 36 B total / ~3 B active | ~18 GB active | MoE — fast |
| `Qwen/Qwen3.5-122B-A10B` | 122 B / ~10 B active | ~40 GB active | MoE — large |

In [ ]:
# ── Qwen 3.5 ───────────────────────────────────────────────────────────────────────────
# Reasoning stays ON. <think>…</think> stripped from verdicts automatically.
# Uncomment one size.

# QWEN35_ID = "Qwen/Qwen3.5-4B"
QWEN35_ID = "Qwen/Qwen3.5-0.8B"       # smallest — smoke tests
# QWEN35_ID = "Qwen/Qwen3.5-9B"
# QWEN35_ID = "Qwen/Qwen3.5-27B"
# QWEN35_ID = "Qwen/Qwen3.5-35B-A3B"    # MoE, ~3B active
# QWEN35_ID = "Qwen/Qwen3.5-122B-A10B"  # MoE, ~10B active

t0 = time.time()
backend = ModelBackend(
    model_name=QWEN35_ID,
    dtype=torch.bfloat16,
    device_map="auto",
    # To hard-disable thinking (faster, less compute):
    # chat_template_kwargs={"enable_thinking": False},
)
print(f"Loaded in {time.time()-t0:.1f}s")
print(f"  hidden_size    = {backend.hidden_size}")
print(f"  num_layers     = {backend.num_layers}")
print(f"  strip_thinking = {backend._strip_thinking}")

writer_backend = editor_backend = backend

# ── Fast-path sanity check (Qwen 3.5 only) ────────────────────────────────────────────
# Warn LOUDLY if the linear-attention CUDA kernels aren't installed — without
# them Qwen 3.5 runs 10-20× slower because DeltaNet layers fall back to a pure
# Python `for` loop.
if "qwen3.5" in QWEN35_ID.lower() or "qwen3_5" in QWEN35_ID.lower():
    try:
        import fla  # flash-linear-attention
        _fla_ok = True
    except ImportError:
        _fla_ok = False
    try:
        import causal_conv1d
        _ccc_ok = True
    except ImportError:
        _ccc_ok = False

    if not (_fla_ok and _ccc_ok):
        print("\n" + "!" * 78)
        print("! WARNING: Qwen 3.5 fast path is NOT available.")
        print(f"!   flash-linear-attention : {'OK' if _fla_ok else 'MISSING'}")
        print(f"!   causal-conv1d          : {'OK' if _ccc_ok else 'MISSING'}")
        print("! The model will run 10-20× slower.")
        print("! Re-run the install cell and restart the runtime, or switch to")
        print("! Qwen 2.5 / LLaMA / Gemma which don't need these kernels.")
        print("!" * 78)
    else:
        print("\nQwen 3.5 fast path: flash-linear-attention + causal-conv1d detected.")

In [ ]:
# ── Qwen 3.5 with 4-bit quantization (fits on a 16 GB GPU) ───────────────
# from transformers import BitsAndBytesConfig
# bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
#                          bnb_4bit_quant_type="nf4")
# backend = ModelBackend("Qwen/Qwen3.5-27B", dtype=torch.bfloat16,
#                        device_map="auto", quantization_config=bnb)
# writer_backend = editor_backend = backend

### 1b — Other supported models

In [ ]:
# ── Other families (uncomment one) ───────────────────────────────────────────────────

# Qwen 2.5
# backend = ModelBackend("Qwen/Qwen2.5-7B-Instruct", dtype=torch.bfloat16, device_map="auto")

# LLaMA 3.3
# backend = ModelBackend("meta-llama/Llama-3.3-70B-Instruct", dtype=torch.bfloat16, device_map="auto")

# Gemma 3
# backend = ModelBackend("google/gemma-3-27b-it", dtype=torch.bfloat16, device_map="auto")

# K2-V2
# backend = K2V2Backend(dtype=torch.bfloat16, device_map="auto")

# writer_backend = editor_backend = backend

### 1c — Cross-model (different writer / editor)

In [ ]:
# writer_backend = ModelBackend("Qwen/Qwen3.5-9B",  dtype=torch.bfloat16, device_map="auto")
# editor_backend = ModelBackend("Qwen/Qwen3.5-27B", dtype=torch.bfloat16, device_map="auto")
# proj = HiddenStateProjector.between(writer_backend, editor_backend)
# print(f"Projection: {proj.src_size} → {proj.tgt_size}  needs={proj.needs_projection}")

## 2 — Sanity check

In [ ]:
t0 = time.time()
ids = writer_backend.build_chat_prompt(
    system="You are a helpful assistant.",
    user="Say hi in 5 words.",
)
r = writer_backend.generate(ids, max_new_tokens=256, capture_hidden=True)
print(f"[{time.time()-t0:.1f}s]")
print(f"output_text   : {r.output_text!r}")
if r.thinking_text:
    print(f"thinking      : {r.thinking_text[:120]}...")

# Verify HS covers ALL answer tokens (needed for full injection into editor)
hs_len    = r.hidden_states[-1].shape[1]
full_len  = r.output_ids.shape[1]
ans_start = r.prompt_len + r.answer_offset
ans_end   = ans_start + len(r.gen_token_ids)
print(f"prompt_len    = {r.prompt_len}")
print(f"answer_offset = {r.answer_offset}  (thinking tokens skipped)")
print(f"answer tokens = {len(r.gen_token_ids)}  positions [{ans_start}, {ans_end})")
print(f"HS length     = {hs_len}  full_len = {full_len}")

assert r.output_text.strip(), "Empty output — check model load or chat template."
assert "<think>" not in r.output_text, "Thinking stripping failed."
assert hs_len == full_len, (
    f"HS length ({hs_len}) < output length ({full_len}). "
    "generate() must capture HS for all tokens (capture_all_tokens=True, default)."
)

## 3 — Build the graph and run all three arms

```
writer → probe_editor            (A: raw text → editor, also captures editor HS)
       → probe_editor_selfhs     (B: editor's own HS re-injected at inject_layer)
       → probe_editor_writerhs   (C: writer's answer HS injected into editor at inject_layer)
       → END
```

**Injection is always all at once:** for each arm, every answer/draft token is replaced in a **single**
vectorised forward pass — not one token at a time. `inject_layer` picks **one** layer (0 = embedding
output, `num_layers//2` = mid-stack, etc.); HS are overwritten at that layer only and the rest of the
forward pass runs normally.

**Cost = exactly 4 `generate()` calls per run.**

In [ ]:
TASK = "Describe a golden retriever in 3 sentences."
MAX_WRITER_TOKENS = 80    # raise to 300+ if thinking is ON and the answer is truncating
MAX_EDITOR_TOKENS = 48
INJECT_LAYER      = 0     # 0 = embedding; try writer_backend.num_layers // 2 for mid-stack
VERBOSE_TIMING    = True  # print wall time per node

graph = make_graph(
    writer_backend,
    editor_backend,
    max_writer_tokens=MAX_WRITER_TOKENS,
    max_editor_tokens=MAX_EDITOR_TOKENS,
    inject_layer=INJECT_LAYER,
    verbose_timing=VERBOSE_TIMING,
)

print(f"Task: {TASK!r}")
t0 = time.time()
final = graph.invoke({"task": TASK})
print(f"Done in {time.time()-t0:.1f}s")

## 4 — Three-arm comparison

In [ ]:
print("══ WRITER OUTPUT (clean — thinking stripped) " + "═"*28)
print(final["writer_output_text"])

wr = final["writer_result"]
if wr.thinking_text:
    print(f"\n  [thinking: {len(wr.thinking_text)} chars, {wr.answer_offset} thinking tokens]")

print("\n── Arm A  (raw text → editor) " + "─"*42)
print(final["editor_verdict"])

print("\n── Arm B  (editor's own HS re-injected) " + "─"*31)
print(final["editor_selfhs_verdict"])

print("\n── Arm C  (writer's HS injected into editor) " + "─"*27)
print(final["editor_writerhs_verdict"])

print()
print("A ≈ B  →  editor's internal repr ≡ surface text")
print("A ≈ C  →  writer's hidden states ≡ writer's text")
print("B ≈ C  →  writer and editor share compatible representations")

## 5 — (Optional) Inspect writer's thinking

The raw chain-of-thought is preserved in `result.thinking_text` even though it is
stripped from verdicts and from the token sequence used for injection.

In [ ]:
wr = final["writer_result"]
if wr.thinking_text:
    print(f"Writer thinking ({wr.answer_offset} tokens):")
    print(wr.thinking_text)
else:
    print("No thinking block (non-Qwen3 model or thinking was empty)")

## 6 — Plug the 3 probe nodes into an existing LangGraph workflow

If you already have a writer node in your graph, you can bolt the 3 probe
arms onto the end of it. Your state schema just needs to extend `ProbeMixin`.

In [ ]:
from langgraph.graph import StateGraph, END

class MyState(ProbeMixin, total=False):
    task: str

def my_writer(state: MyState) -> MyState:
    prompt_ids = writer_backend.build_chat_prompt(
        system="You are a helpful writing assistant.",
        user=state["task"],
    )
    result = writer_backend.generate(prompt_ids, max_new_tokens=80, capture_hidden=True)
    return {
        **state,
        "writer_result": result,
        "writer_output_text": result.output_text,         # clean, thinking stripped
        "writer_output_token_ids": result.gen_token_ids,  # answer tokens only
    }

g = StateGraph(MyState)
g.add_node("my_writer", my_writer)
g.set_entry_point("my_writer")
add_probe_to_graph(
    g,
    writer_backend,
    editor_backend,
    entry_node="my_writer",
    inject_layer=0,
)
app = g.compile()

out = app.invoke({"task": "Explain why the sky is blue in two sentences."})
print("Arm A:", out["editor_verdict"])
print("Arm B:", out["editor_selfhs_verdict"])
print("Arm C:", out["editor_writerhs_verdict"])

## 7 — Save outputs (optional)

In [ ]:
import os

out_dir = "outputs"
os.makedirs(out_dir, exist_ok=True)

wr = final["writer_result"]
with open(f"{out_dir}/arm_verdicts.txt", "w", encoding="utf-8") as f:
    f.write(f"model        : {writer_backend.model_name}\n")
    f.write(f"task         : {TASK}\n")
    f.write(f"inject_layer : {INJECT_LAYER}\n")
    f.write(f"thinking_len : {wr.answer_offset} tokens\n\n")
    if wr.thinking_text:
        f.write(f"=== Writer thinking ===\n{wr.thinking_text}\n\n")
    f.write(f"=== Writer output (clean) ===\n{final['writer_output_text']}\n\n")
    f.write(f"=== Arm A: editor_verdict (raw text) ===\n{final['editor_verdict']}\n\n")
    f.write(f"=== Arm B: editor_selfhs_verdict (editor's own HS) ===\n{final['editor_selfhs_verdict']}\n\n")
    f.write(f"=== Arm C: editor_writerhs_verdict (writer's HS) ===\n{final['editor_writerhs_verdict']}\n")

print(f"Saved to ./{out_dir}/arm_verdicts.txt")